# 01 — Data Exploration

Exploring the **synthetic** chilli dataset that LotIQ trains on.

> Reminder: this is synthetic prototype data from a domain-informed
> simulator, **not** real operational data.

In [ ]:
# Make the `lotiq` package importable whether Jupyter launched from the repo
# root or the notebooks/ folder.
import sys, pathlib
try:
    import lotiq  # noqa: F401
except ModuleNotFoundError:
    for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_p / "src" / "lotiq").exists():
            sys.path.insert(0, str(_p / "src")); break
    import lotiq  # noqa: F401
print("lotiq", lotiq.__version__)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lotiq.config import SYNTHETIC_CSV, CHILLI, MODEL_FEATURES, TARGET

df = pd.read_csv(SYNTHETIC_CSV)
print(df.shape)
df.head()

## Target distribution

How risk scores and risk bands are spread across the warehouse.

In [ ]:
display(df[TARGET].describe().to_frame().T)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(df[TARGET], bins=30, color='#3b6fb0', edgecolor='white')
ax[0].set_title('Risk score distribution'); ax[0].set_xlabel('risk_score')

order = ['healthy', 'moderate', 'high', 'critical']
counts = df['risk_level'].value_counts().reindex(order).fillna(0)
colours = ['#2e9e5b', '#e6b800', '#e67300', '#d13438']
ax[1].bar(order, counts.values, color=colours)
ax[1].set_title('Lots per risk band')
plt.tight_layout(); plt.show()

## Which features move risk?

Correlation of each feature with the risk score. Signs should match the
domain: moisture / humidity / temperature / CO2 / storage push risk **up**;
oleoresin and colour push it **down**.

In [ ]:
num = [c for c in MODEL_FEATURES]
corr = df[num + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
colours = ['#2e9e5b' if v < 0 else '#d13438' for v in corr.values]
plt.figure(figsize=(8, 4))
plt.barh(corr.index, corr.values, color=colours)
plt.title('Correlation with risk_score'); plt.axvline(0, color='k', lw=0.8)
plt.tight_layout(); plt.show()
corr

## Feature relationships

Moisture vs risk (coloured by intake grade) and storage time vs risk.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
grade_colour = {'A': '#2e9e5b', 'B': '#e6b800', 'C': '#d13438'}
for g, sub in df.groupby('initial_quality_grade'):
    ax[0].scatter(sub['moisture_content'], sub['risk_score'], s=8,
                  alpha=0.4, label=f'Grade {g}', color=grade_colour.get(g))
ax[0].axvline(CHILLI.thresholds['moisture_content'], color='k', ls='--', lw=1)
ax[0].set_xlabel('moisture_content (%)'); ax[0].set_ylabel('risk_score')
ax[0].set_title('Moisture vs risk'); ax[0].legend()

ax[1].scatter(df['storage_days'], df['risk_score'], s=8, alpha=0.3, color='#3b6fb0')
ax[1].set_xlabel('storage_days'); ax[1].set_ylabel('risk_score')
ax[1].set_title('Storage time vs risk')
plt.tight_layout(); plt.show()

**Takeaways**

- No single feature perfectly determines risk — the label carries genuine,
  irreducible variation by design (see `docs/architecture.md`).
- Worse intake grades (C) sit higher up the risk axis at the same moisture.
- Moisture and colour/oleoresin are the strongest signals, but several
  features contribute — which is what makes per-lot SHAP explanations useful.